In [1]:
%load_ext sql

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

In [6]:
import polars as pl

%config SqlMagic.autopolars = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 50

# With autopolars, the notebook renders Polars frames — show up to 50 rows
# (smaller results are shown in full).
pl.Config.set_tbl_rows(50)

polars.config.Config

In [7]:
import duckdb

# In-memory DuckDB; data files are read in SQL via relative paths (cwd = notebooks/).
conn = duckdb.connect()
%sql conn --alias duckdb

Convert `asin2category.json` (one giant object map) into Parquet so DuckDB can scan it lazily.

In [10]:
%run ../scripts/convert_json_parquet.py

skip convert — already exists: ../data/amazon_musical_instruments/asin2category.parquet


In [11]:
%%sql
-- Register paths once as views (lazy: no data copy). Then query by short name.
CREATE OR REPLACE VIEW reviews AS
SELECT * FROM read_ndjson('../data/amazon_musical_instruments/Musical_Instruments.jsonl');

CREATE OR REPLACE VIEW meta_Musical_Instruments AS
SELECT * FROM read_ndjson('../data/amazon_musical_instruments/meta_Musical_Instruments.jsonl');

CREATE OR REPLACE VIEW full_products AS
SELECT * FROM '../data/amazon_musical_instruments/full-*.parquet';

CREATE OR REPLACE VIEW asin2category AS
SELECT * FROM '../data/amazon_musical_instruments/asin2category.parquet';

/Users/viktorhristovski/repos/uni/nlp/projects/llm-knowledge-enhancement/.venv/lib/python3.13/site-packages/sql/run/resultset.py:527: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  return getattr(native_connection, converter_name)()


Count
i64


In [12]:
%%sql 
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'main'
  AND table_name = 'reviews';

column_name,data_type
str,str
"""rating""","""DOUBLE"""
"""title""","""VARCHAR"""
"""text""","""VARCHAR"""
"""images""","""STRUCT(small_image_url VARCHAR…"
"""asin""","""VARCHAR"""
"""parent_asin""","""VARCHAR"""
"""user_id""","""VARCHAR"""
"""timestamp""","""BIGINT"""
"""helpful_vote""","""BIGINT"""


In [13]:
%%sql 
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'main'
  AND table_name = 'meta_Musical_Instruments';

column_name,data_type
str,str
"""main_category""","""VARCHAR"""
"""title""","""VARCHAR"""
"""average_rating""","""DOUBLE"""
"""rating_number""","""BIGINT"""
"""features""","""VARCHAR[]"""
"""description""","""VARCHAR[]"""
"""price""","""JSON"""
"""images""","""STRUCT(thumb VARCHAR, ""large"" …"
"""videos""","""STRUCT(title VARCHAR, url VARC…"


In [14]:
%%sql 
select count(*) from meta_Musical_Instruments; 

count_star()
i64
213593


In [15]:
%%sql 
select count(*)/2 from meta_Musical_Instruments; 

(count_star() / 2)
f64
106796.5
